# Market price curves

One panel per resource showing `price(inventory)` over the range `[0, 2*I0]`. The base price is marked at `I0=10000`; vertical dashed lines at `I0 - T` and `I0 + T` bracket the anchor throughput window. The price floor is $1.

Below `I0` the sell price rises as inventory falls (players buying, town consuming). Above `I0` the sell price falls as inventory grows (players selling). Different shape functions and target multipliers make different resources behave very differently on each side of `I0`.

Formulas match `kaggle-environments >= 1.32.7` (post-Aug rebalance).

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from kaggriculture.env.constants import HINGE_GAIN, MARKET_I0, MARKET_PARAMS, PRICE_FLOOR

FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def shape(func: str, x: float, T: float | None = None) -> float:
    x = max(0.0, x)
    if func == "linear":
        return x
    if func == "sq":
        return x * x
    if func == "sqrt":
        return math.sqrt(x)
    if func == "log":
        return math.log(1.0 + x)
    if func == "log10":
        return math.log10(1.0 + x)
    if func == "hinge":
        if not T or T <= 0:
            return x
        u = x / T
        return u + HINGE_GAIN * max(0.0, u - 1.0) ** 2
    raise ValueError(f"unknown shape {func!r}")


def market_price(item: str, inv: float) -> int:
    p = MARKET_PARAMS[item]
    base = p["base"]
    I0 = p["I0"]
    T = p["T"]
    if inv < I0:
        f = p["below_func"]
        amp = p["below_target"] * base / shape(f, T, T)
        price = base + amp * shape(f, I0 - inv, T)
    else:
        f = p["above_func"]
        amp = p["above_target"] * base / shape(f, T, T)
        price = base - amp * shape(f, inv - I0, T)
    return max(PRICE_FLOOR, round(price))

In [ ]:
products = list(MARKET_PARAMS)
inv_axis = np.arange(0, 2 * MARKET_I0 + 1, 20)

fig, axes = plt.subplots(3, 3, figsize=(14, 11), sharex=True)
for ax, product in zip(axes.flat, products, strict=True):
    prices = [market_price(product, int(x)) for x in inv_axis]
    ax.plot(inv_axis, prices, color="#1f77b4")
    p = MARKET_PARAMS[product]
    ax.axvline(p["I0"], color="black", linestyle=":", linewidth=0.8, label="I0")
    ax.axvline(p["I0"] - p["T"], color="gray", linestyle="--", linewidth=0.7, label="I0 ± T")
    ax.axvline(p["I0"] + p["T"], color="gray", linestyle="--", linewidth=0.7)
    ax.axhline(p["base"], color="green", linestyle=":", linewidth=0.8, label="base")
    ax.set_title(
        f"{product}  base=${p['base']}  T={p['T']}\n"
        f"below={p['below_func']}({p['below_target']}), above={p['above_func']}({p['above_target']})",
        fontsize=9,
    )
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)

for ax in axes[-1]:
    ax.set_xlabel("market inventory")
for ax in axes[:, 0]:
    ax.set_ylabel("sell price ($, log)")

fig.suptitle("Kaggriculture sell price curves (v1.32.7)", y=1.00, fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "market-curves-grid.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
rows = []
for product, p in MARKET_PARAMS.items():
    I0 = p["I0"]
    T = p["T"]
    base = p["base"]
    rows.append(
        {
            "product": product,
            "base": base,
            "scarcity_1T": market_price(product, I0 - T),
            "scarcity_5T": market_price(product, I0 - 5 * T),
            "glut_1T": market_price(product, I0 + T),
            "glut_5T": market_price(product, I0 + 5 * T),
        }
    )

import pandas as pd

df = pd.DataFrame(rows).set_index("product")
df["scarcity_1T_pct"] = (df["scarcity_1T"] / df["base"] - 1) * 100
df["glut_1T_pct"] = (df["glut_1T"] / df["base"] - 1) * 100
df["scarcity_5T_pct"] = (df["scarcity_5T"] / df["base"] - 1) * 100
df["glut_5T_pct"] = (df["glut_5T"] / df["base"] - 1) * 100
df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(df))
width = 0.35
ax.bar(
    x - width / 2, df["scarcity_1T_pct"], width, label="inv = I0 - T (scarcity)", color="#d62728"
)
ax.bar(x + width / 2, df["glut_1T_pct"], width, label="inv = I0 + T (glut)", color="#2ca02c")
ax.set_xticks(x)
ax.set_xticklabels(df.index, rotation=30, ha="right")
ax.set_ylabel("price change vs base (%)")
ax.set_title("Response at one T of inventory shift")
ax.axhline(0, color="black", linewidth=0.8)
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "market-sensitivity-1T.png", dpi=140, bbox_inches="tight")
plt.show()

## What the shapes mean strategically

Premium goods (strawberry, melon, milk, wool) all have `above_target > 1` with `sq` or `linear` shapes: modest gluts drive them to the price floor. Bundling and timing matters more than raw volume for these.

Carrot, tomato, and egg use the `hinge` shape below the knee: prices stay near base under ordinary demand and spike sharply once demand runs past `T`. The town shops that consume them (pet cafe, pizza shop, bakery, brunch spot) can push each past `T` in a nonzero fraction of seasons.

Wheat is the staple: it panics on scarcity (`sqrt`, `below_target=0.80`) and absorbs gluts gently (`log`, `above_target=0.20`). Combined with animal feed demand, wheat is the only reliable-price resource in the game.

Fertilizer uses symmetric `linear` on both sides and is one of two products (with wheat) available via `BUY_PRODUCT`.